# Tome 10, Chapter 5: Infinitely divisible and stable laws (Python companion)

Exercises first, solutions at the end. Simulating stable laws, averages that refuse to settle, a domain of attraction checked through the characteristic function, the fair gambler's waiting time, Holtsmark's field, and a computer science application: **robust mean estimation for heavy-tailed data** (latencies, file sizes, transaction amounts) with the median of means. Requires `numpy` and `scipy`.

In [ ]:
import math
import numpy as np
from scipy import stats, special, integrate
rng = np.random.default_rng(50)

def sym_stable(alpha, size):
    # Chambers-Mallows-Stuck (1976), symmetric case: characteristic function exp(-|t|^alpha)
    V = rng.uniform(-math.pi / 2, math.pi / 2, size)
    W = rng.exponential(1.0, size)
    if alpha == 1:
        return np.tan(V)
    return (np.sin(alpha * V) / np.cos(V)**(1 / alpha)) * (np.cos((1 - alpha) * V) / W)**((1 - alpha) / alpha)

## Exercises

**P5.1 (★) Simulating stable laws.** The function `sym_stable` implements the Chambers–Mallows–Stuck method. Check it: for $\alpha=0.5,1,1.5,2$, compare the empirical characteristic function $\frac1N\sum_k\cos(tX_k)$ with $e^{-|t|^\alpha}$ at a few $t$. (For $\alpha=2$, what is the variance?)

**P5.2 (★) Averages that refuse to settle.** Plot or tabulate the interquartile range of the average of $n$ i.i.d. copies, for $n=1,10,100,1000$ and $\alpha=0.5,1,1.5,2$. Compare with Exercise 5.3: the spread scales like $n^{1/\alpha-1}$.

**P5.3 (★★) A domain of attraction.** Let $X$ be symmetric Pareto: $P(|X|>x)=x^{-3/2}$ for $x\ge1$. (a) Compute $\kappa_{3/2}=\frac32\int_0^\infty(1-\cos u)u^{-5/2}\dd u$ numerically and compare with $\Gamma(1-\alpha)\cos\frac{\pi\alpha}2$ at $\alpha=\frac32$. (b) Check Theorem 5.8: the empirical characteristic function of $S_n/n^{2/3}$, for $n=10,100,1000$, against $e^{-\kappa|t|^{3/2}}$.

**P5.4 (★★) The fair gambler.** (a) Simulate $10^5$ waiting times $\tau_1$ of the coin-tossing walk to reach $+1$, capped at $10^6$ steps, and compare $P(\tau_1>k)$ with $\sqrt{2/(\pi k)}$. (b) Using the identity $\tau_n=\tau_1^{(1)}+\dots+\tau_1^{(n)}$, compare the law of $\tau_n/n^2$ for $n=50$ with the Lévy law $P(T_1\le t)=2(1-\Phi(t^{-1/2}))$. (c) What is the median of $\tau_1$? Its mean?

**P5.5 (★★) Holtsmark's field in one dimension.** Place charges at the points of a Poisson process of rate $1$ on $[-R,R]$, with random signs and force $\pm1/|x|$ ($p=1$). By Exercise 5.7 the total force tends to a symmetric stable law of index $1$, a Cauchy law. Which scale? Check by simulation with $R=10^4$.

**P5.6 (★★★) CS application: robust means.** Request latencies of a web service are often heavy-tailed. Model them by $L=1+P$ with $P$ Pareto: $P(P>x)=x^{-1.5}$ for $x\ge1$, so the mean $\mathbb EL=1+3=4$ exists but the variance is infinite. With $n=1000$ measurements, compare three estimators of the mean over $10^4$ repetitions: the empirical mean, a $5\%$-trimmed mean, and the **median of means** with $k=20$ blocks. Report the bias, the $95\%$ quantile of the absolute error, and the probability of an error above $1$. Then prove the guarantee behind the median of means: if each block mean is within $\varepsilon$ of $\mu$ with probability at least $\frac34$, the median of $k$ independent blocks fails with probability at most $e^{-k/8}$ (Hoeffding on the number of bad blocks, Tome 3, Theorem 4.9).

## Solutions

### P5.1
**How to find it.** The empirical characteristic function converges at rate $N^{-1/2}$ whatever the tails, because $|\cos|\le1$: it is the right tool for laws without moments. For $\alpha=2$, $e^{-t^2}$ is the characteristic function of $N(0,2)$.

In [ ]:
N = 10**6
for alpha in (0.5, 1.0, 1.5, 2.0):
    X = sym_stable(alpha, N)
    row = [f"t={t}: {np.mean(np.cos(t * X)):.4f} vs {math.exp(-abs(t)**alpha):.4f}" for t in (0.5, 1.0, 2.0)]
    extra = f"   variance {X.var():.3f}" if alpha == 2 else ""
    print(f"alpha={alpha}: " + ";  ".join(row) + extra)

Agreement to about $10^{-3}$, the Monte Carlo error. For $\alpha=2$ the formula reduces to $\sin(2V)/\cos(V)^{1/2}\cdot(\cos V/W)^{-1/2}=2\sin V\sqrt W$, a classical way to produce $N(0,2)$ from an angle and an exponential (it is the Box–Muller transform in disguise).

### P5.2
**How to find it.** The interquartile range exists for every law, unlike the standard deviation. By Exercise 5.3, $S_n/n$ has the law of $n^{1/\alpha-1}X$, so its interquartile range is exactly $n^{1/\alpha-1}$ times that of $X$.

In [ ]:
reps = 20000
print("alpha   n=1      n=10     n=100    n=1000    predicted ratio n=1000")
for alpha in (0.5, 1.0, 1.5, 2.0):
    iqr = []
    for n in (1, 10, 100, 1000):
        m = sym_stable(alpha, (reps // max(1, n // 100), n)).mean(axis=1) if n > 1 else sym_stable(alpha, reps)
        q = np.quantile(m, [0.25, 0.75]); iqr.append(q[1] - q[0])
    print(f"{alpha:4}  " + "  ".join(f"{v:8.3g}" for v in iqr) + f"   {1000**(1/alpha-1):10.3g} (observed {iqr[-1]/iqr[0]:.3g})")

For $\alpha=\frac12$ the average of $1000$ observations is a million times more spread out than one observation, for $\alpha=1$ the spread does not change, and only for $\alpha>1$ does averaging help, slowly ($1000^{-1/3}=0.1$ for $\alpha=\frac32$, against $1000^{-1/2}\approx0.03$ for the Gaussian).

### P5.3
**How to find it.** (a) Integrate numerically, splitting at $1$ (near $0$ the integrand behaves like $\frac12u^{-1/2}$, integrable). (b) Sample $|X|=U^{-2/3}$ with $U$ uniform, and a random sign.

In [ ]:
alpha = 1.5
f = lambda u: (1 - math.cos(u)) * u**(-1 - alpha)
g = lambda u: u**(-1 - alpha)
tail = integrate.quad(g, 1, np.inf)[0] - integrate.quad(g, 1, np.inf, weight='cos', wvar=1)[0]   # oscillatory part done by QAWF
kappa = alpha * (integrate.quad(f, 0, 1)[0] + tail)
print(f"(a) kappa = {kappa:.6f},  Gamma(1-alpha) cos(pi alpha/2) = {special.gamma(1-alpha)*math.cos(math.pi*alpha/2):.6f}")
for n in (10, 100, 1000):
    reps = 20000
    X = rng.uniform(size=(reps, n))**(-1 / alpha) * rng.choice([-1, 1], size=(reps, n))
    Z = X.sum(axis=1) / n**(1 / alpha)
    print(f"(b) n={n:5d}: " + "  ".join(f"t={t}: {np.mean(np.cos(t*Z)):.4f} vs {math.exp(-kappa*t**alpha):.4f}" for t in (0.3, 0.7, 1.5)))

(a) Both give $2.5066$, which happens to be $\sqrt{2\pi}$ (since $\Gamma(-\frac12)=-2\sqrt\pi$ and $\cos\frac{3\pi}4=-\frac1{\sqrt2}$); the identity $\kappa_\alpha=\Gamma(1-\alpha)\cos\frac{\pi\alpha}2$ holds for all $\alpha\in(0,2)$, $\alpha\ne1$ (with $\kappa_1=\frac\pi2$), and is proved by contour integration. (b) The fit improves with $n$, slowly at small $t$: the convergence to a stable law is typically slower than the Berry–Esseen rate.

### P5.4
**How to find it.** Simulate the walks in a vectorised loop, removing the walkers who have arrived. For (b), sum $n$ independent waiting times; capped values are rare but must not be discarded silently (we flag them).

In [ ]:
walkers, cap = 10**5, 10**6
pos = np.zeros(walkers, dtype=np.int64); t_arr = np.full(walkers, -1, dtype=np.int64)
alive = np.arange(walkers); step = 0
while alive.size and step < cap:
    step += 1
    pos[alive] += rng.choice(np.array([-1, 1]), size=alive.size)
    done = pos[alive] == 1
    t_arr[alive[done]] = step
    alive = alive[~done]
print(f"(a) not arrived after {cap} steps: {alive.size} (predicted {walkers*math.sqrt(2/(math.pi*cap)):.0f})")
for k in (10, 100, 1000, 10000, 100000):
    emp = np.mean((t_arr > k) | (t_arr < 0))
    print(f"    P(tau_1 > {k:6d}) = {emp:.4f}   sqrt(2/(pi k)) = {math.sqrt(2/(math.pi*k)):.4f}")

In [ ]:
n = 50
pool = t_arr.astype(float); pool[pool < 0] = np.inf
sums = pool[rng.integers(0, walkers, size=(20000, n))].sum(axis=1) / n**2
for t in (0.5, 1, 2, 5, 20):
    print(f"(b) P(tau_50/2500 <= {t:4}) = {np.mean(sums <= t):.4f}   Levy: {2*(1-stats.norm.cdf(t**-0.5)):.4f}")
fin = t_arr[t_arr > 0]
print(f"(c) P(tau_1 = 1) = {np.mean(t_arr == 1):.4f};  mean of the finite ones = {fin.mean():.0f}, vs sqrt(2 cap/pi) = {math.sqrt(2*cap/math.pi):.0f}")

(a) The tail matches $\sqrt{2/(\pi k)}$ already at $k=100$ (for odd $k$ the exact value is $P(S_1\le0,\dots,S_k\le0)$, a central binomial coefficient, which has exactly this asymptotics by Stirling). (b) The fit with the Lévy law is good to about $0.01$: the identity of Theorem 5.11 at work. The simulated probabilities are slightly higher than the limit; this is a finite-$n$ effect of the discrete walk, which shrinks as $n$ grows. (c) Half the time the first toss wins, so the median is $1$. But the mean of the waits that ended before the cap is about $\sqrt{2\,\text{cap}/\pi}\approx800$ (integrate the tail $\sqrt{2/(\pi k)}$ up to the cap and subtract the capped part), and it grows without bound with the cap: the mean is infinite. Resampling from a capped pool slightly underestimates the tail in (b); a cap of $10^6$ steps against a scale of $n^2=2500$ makes this invisible.

### P5.5
**How to find it.** By the computation of Exercise 5.7 with $p=1$, the limit exponent is $-2\rho\int_0^\infty(1-\cos(t/x))\dd x=-2\rho|t|\int_0^\infty(1-\cos u)u^{-2}\dd u=-\pi\rho|t|$: a Cauchy law of scale $\pi\rho$. Simulate the number of points, place them uniformly, sum the signed forces.

In [ ]:
R, reps = 10**4, 4000
F = np.empty(reps)
for r in range(reps):
    m = rng.poisson(2 * R)
    x = rng.uniform(-R, R, m)
    F[r] = np.sum(rng.choice([-1.0, 1.0], m) / np.abs(x))
q = np.quantile(F, [0.25, 0.5, 0.75])
print(f"quartiles {np.round(q, 3)}; a Cauchy law of scale pi has quartiles {np.round([-math.pi, 0, math.pi], 3)}")
print("KS distance to Cauchy(scale pi):", round(stats.kstest(F / math.pi, 'cauchy').statistic, 4))

The quartiles of a Cauchy law of scale $s$ are $\pm s$, and the simulation gives $\pm\pi$: one-dimensional "Coulomb" charges with force $1/r$ produce a Cauchy field. The nearest charge dominates: with density $1$, the nearest is at distance about $\frac12$, giving forces of order $2$, and occasionally a charge sits very close and the force is huge, which is the heavy tail.

### P5.6
**How to find it.** The median of means splits the sample into $k$ blocks, averages each, and returns the median of the block means. For the proof: if the median is more than $\varepsilon$ away from $\mu$, at least half the blocks are bad, while each block is bad with probability at most $\frac14$; the number of bad blocks is binomial, and Hoeffding gives $P(B\ge\frac k2)\le e^{-2k(1/4)^2}=e^{-k/8}$. By Chebyshev, a block of size $n/k$ is good with probability $\ge\frac34$ when $\varepsilon\ge2\sigma\sqrt{k/n}$, so with finite variance the median of means achieves sub-Gaussian error bars; with infinite variance, as here, the same argument works with a Marcinkiewicz–Zygmund or truncation bound in place of Chebyshev.

In [ ]:
n, reps, k = 1000, 10000, 20
L = 1 + rng.uniform(size=(reps, n))**(-1 / 1.5)
mean = L.mean(axis=1)
trim = stats.trim_mean(L, 0.05, axis=1)
mom = np.median(L.reshape(reps, k, n // k).mean(axis=2), axis=1)
for name, est in (("mean", mean), ("5% trimmed mean", trim), ("median of means (k=20)", mom)):
    err = est - 4
    print(f"{name:24s} bias {err.mean():+.3f}   95% |error| {np.quantile(np.abs(err), 0.95):.3f}   P(|error|>1) {np.mean(np.abs(err) > 1):.4f}")

The empirical mean is unbiased but has occasional huge errors: a single enormous latency drags it, which is the stable limit of Theorem 5.10 at work (fluctuations of order $n^{-1/3}$, not $n^{-1/2}$, with heavy tails). The trimmed mean is very stable but *biased*: it throws away the large values that carry part of the mean, and its bias does not vanish. The median of means has a negative bias ($-0.39$ here: the block means are skewed to the right, and their median sits below their mean), but its errors never exceeded $1$ in $10^4$ trials, against $3\%$ of the time for the mean. Which to report depends on the question: for capacity planning the rare huge latencies matter and you want the mean *and* the tail; for comparing two versions of a service, a robust estimator with honest error bars is better. Robust statistics of this kind (median of means, Catoni's estimator) are a whole research field in learning theory.